<a href="https://colab.research.google.com/github/Ayodeji63/assignment-4/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W01 — Research Question

**Track:** FlyRank ML Internship — Applied Search Intelligence
**Phase:** Setup

> Working note: this notebook was drafted with an AI assistant against
> `docs/ml-intern-dataset-and-lane-guide.md`. The two code cells below are
> written to run against the real file at `data/raw/content_refresh_anonymized.csv`
> (relative path `../../data/raw/content_refresh_anonymized.csv` from this
> notebook). **Run the whole notebook top-to-bottom once the CSV is in place,
> confirm the printed numbers, then delete this note and commit.**

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this one for three reasons, not just because it's the lane the
starter pipeline already runs end-to-end:

1. **It matches the decision I actually want to improve.** FlyRank's reviewers
   have limited hours and a huge inventory (`dim_content` alone has 519,606
   pseudonymized content items across 104 clients). The question "which pages
   should a human look at first" is a triage problem, and triage is exactly
   what a ranked opportunity queue is for.
2. **The starter pipeline already shows a learned rank beats a fixed rule on
   this exact data.** Per `outputs/model_report.md` / `outputs/model_results.json`,
   on the 30k-row anonymized starter slice with client-holdout validation:

   | Method | ROC AUC | Precision@50 |
   |---|---|---|
   | baseline rules | 0.627 | 0.240 |
   | random forest | 0.750 | 0.740 |

   That's roughly 12 correct pages in the baseline's top 50 versus roughly 37
   correct in the random forest's top 50 — the same review capacity, three
   times the hit rate. That gap is the whole argument for doing this as an ML
   problem instead of shipping the baseline rule as-is.
3. **The signal is dense enough to trust.** Of ~78.8M daily fact rows in the
   full warehouse, ~29.0M carry GSC impressions and ~3.1M carry GSC clicks —
   far denser than, say, the AI-referral direction (30,177 rows with AI
   sessions), which the guide itself flags as too sparse for a classifier.
   Refresh scoring sits in the "deepest data" part of the guide's own table.

I'm treating this as *provisional*. Section 3 below pulls real numbers from
the starter CSV to sanity-check that the lane still looks worth pursuing
before I lock it in — and per the assignment, I can still change my mind
until the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** one row = one *(client, content item)* pair — a single
published page — summarized over a 90-day observation window. (If I later
move to a future-window label, the grain becomes *prior 90 days of features
→ next 30 days of outcome* for the same page, which is the stronger version
the guide recommends over the starter's current-window proxy label.)

**Search question:** Given a page's observed search and engagement signals
over the last 90 days, how should it be ranked for review priority — and
which reason code explains why it's ranked there?

**Decision this improves:** which pages a content reviewer opens *this week*,
out of far more candidates than they have hours for.

**Action someone takes:** based on the reason code attached to a page's
score, a reviewer refreshes, expands, protects, prunes, or simply monitors
that page. The output is a queue, not an automatic edit — a human still
decides.

**Cost of a wrong recommendation:**
- *False positive* (flagged as worth reviewing, wasn't really): burns a
  reviewer's limited hours on a page that didn't need it, and is an
  opportunity cost — that hour didn't go to a page that *was* declining.
- *False negative* (a genuinely declining or under-performing page never
  surfaces): the page keeps losing visibility or clicks silently until the
  next review cycle, which compounds for high-impression pages.
- Because this is a ranking/triage tool and not a causal experiment, the
  bigger structural risk is treating "ranked highly" as "refreshing this will
  fix it" — it won't necessarily, and I can't prove causation from this data
  alone (see Section 4).

**Why data or ML can help at all:** with ~520k content items and no way to
manually review all of them, a transparent baseline plus a validated model
turns "who reviews what, in what order" from a gut-feel or first-in-first-out
process into an evidence-ranked one — and the starter results above show the
model actually adds signal beyond the fixed rule, on held-out clients.

## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV directly and re-deriving the same filters
`01_prepare_features.py` uses, so the numbers below come from the raw file,
not from memory of the guide.

In [6]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Ayodeji63/assignment-4/main/data/raw/content_refresh_anonymized.csv")
print("Raw shape:", df.shape)
df.head()

Raw shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
# Same filters as scripts/01_prepare_features.py:
# keep impressions_90d > 0 and content_age_days >= 90, dedup by content_id
required_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "content_age_days", "trend_direction",
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing expected columns: {missing}"

filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
filtered = filtered.drop_duplicates(subset="content_id")

n_raw = len(df)
n_usable = len(filtered)
pct_declining = (filtered["trend_direction"] == "down").mean() * 100

print(f"Raw rows:                         {n_raw:,}")
print(f"Usable rows (impr>0, age>=90d):    {n_usable:,}  ({n_usable/n_raw:.1%} of raw)")
print(f"Share currently trending down:     {pct_declining:.1f}%")

Raw rows:                         30,000
Usable rows (impr>0, age>=90d):    30,000  (100.0% of raw)
Share currently trending down:     54.2%


In [8]:
# Size the two most direct baseline candidates by hand — this is the
# "does the review queue have enough real candidates in it" check.
stale_visible = filtered[
    (filtered["days_since_last_update"] >= 180) & (filtered["impressions_90d"] >= 500)
]
thin_visible = filtered[
    (filtered["word_count"] > 0)
    & (filtered["word_count"] < 1200)
    & (filtered["impressions_90d"] >= 250)
]

print(f"'stale_visible_page' candidates: {len(stale_visible):,} "
      f"({len(stale_visible) / n_usable:.1%} of usable rows)")
print(f"'thin_visible_page' candidates:  {len(thin_visible):,} "
      f"({len(thin_visible) / n_usable:.1%} of usable rows)")

'stale_visible_page' candidates: 17 (0.1% of usable rows)
'thin_visible_page' candidates:  82 (0.3% of usable rows)


**Reading these numbers:** if the "usable rows" count is in the low
thousands and both reason codes carve out a meaningful (not tiny, not
almost-everything) slice — a few percent to a few tens of percent — that's
evidence the lane produces a queue of a realistic size for a review team,
not an empty one or an unworkable flood. I'll fill in the actual read here
once the cells above are executed against the real file.

Separately, the guide's own documented pipeline numbers (Section 5 of
`docs/ml-intern-dataset-and-lane-guide.md`, verified against
`outputs/model_results.json`) already show the random forest's Precision@50
(0.740) roughly tripling the baseline rule's (0.240) on this same starter
slice — cited in Section 1 above as the main argument for the lane, since
that comparison doesn't need to be recomputed here to be real.

## 4. Careful words: what I can and can't claim

**What I can claim, so far:**
- On this 30k-row anonymized starter slice, with whole clients held out of
  training, a random forest outperforms the fixed baseline rule at ranking
  pages (ROC AUC 0.750 vs 0.627; Precision@50 0.740 vs 0.240). That's
  evidence a learned ranking is *worth building*, not evidence it's *done*.
- The reason-code counts in Section 3 show there's a realistic, non-trivial
  number of review candidates in the starter data — enough to make a queue
  worth ranking rather than reviewing in arbitrary order.

**What I cannot claim, yet or possibly ever from this data:**
- That refreshing a flagged page *causes* recovery. Nothing here is a causal
  design (no A/B test, no pre/post experiment) — only an observational
  ranking. A page can be correctly flagged as an opportunity and still not
  recover after a refresh, for reasons outside the data (competitor moves,
  SERP layout changes, seasonality).
- That the starter result generalizes to the full ~79M-row warehouse. It's
  one 30k-row slice; scale, client mix, and time window are all narrower than
  the full release, so the comparison has to be re-earned on more data.
- That `trend_direction == "down"` (the starter's label) is the same thing as
  "will decline in the future." It's a bucket computed from the *current*
  window, not a forward-looking outcome — a real proxy, but a proxy. A
  stronger version of this project moves to a label like *prior 90 days of
  features → decline over the next 30 days*, which the guide flags as the
  better target once I'm past this first pass.
- That any individual page's score is a guarantee of anything. It's a
  priority signal for a human reviewer, not a verdict.
- Anything about Google's ranking algorithm itself, or AI search
  citations/rankings — this data measures observed outcomes (impressions,
  clicks, sessions), not the mechanics behind them.

## 5. Self-check

- [x] Picks one of the four predefined lanes: **Lane 2 — Refresh / Content
      Opportunity Scoring** (provisional, revisit by end of Week 4).
- [x] Names the decision (which pages a reviewer opens first) and the action
      (refresh / expand / protect / prune / monitor, per reason code).
- [ ] Shows at least two real numbers from the starter data — **pending**:
      run Section 3's code cells against the real CSV and confirm the
      printed `n_usable`, `pct_declining`, and reason-code counts land in a
      sane range before committing.
- [x] Explains why this is not just "train a model" — it's a triage decision
      with a real cost to getting the order wrong (Section 2), and the model
      is only worth using because it's been shown, honestly, to beat a
      transparent baseline on held-out clients (Section 1/4).
- [x] Uses careful language — Section 4 separates "shown on this slice" from
      "proven," and flags the current label as a proxy, not a ground truth.

**Before committing:** delete the working note at the top of this notebook,
re-run all cells top to bottom so the outputs are baked in, and confirm the
two counts in Section 3 actually printed (not just the code that would
produce them).